# EFG point-charge

Here we reproduce results discussed [here](https://github.com/apdioguardi/EFG_point_charge_lattice_sum/tree/main/EFG_point_charge_aAlO2).

In [1]:
import numpy as np

In [2]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent.parent.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT)) 

from muonscripts.constants import constants
from muonscripts.io_tools.read import read_cif
from muonscripts.io_tools.read import read_from_file

from muonscripts.pcefg.lattice import get_atom_kinds 
from muonscripts.pcefg.point_charge import PointChargeEFG
from muonscripts.pcefg.point_charge import compute_efg, point_charge_EFG

In [3]:
import os
dpath='./'

In [4]:
filename = "Ishizawa_1980_aAl2O3_300K_EntryWithCollCode10425.cif"  # 'read_cif' ERROR: CIF File not structured well, converted to VASP format below
filename = "Ishizawa_1980_aAl2O3_300K_EntryWithCollCode10425.vasp"
filename = os.path.join(dpath, filename)
atoms = read_cif(filename) # or with read_from_file
# atoms = read_from_file(filename) # you can parse **read_kwargs depending on the type of the I/O

In [5]:
gamma_sternheimer = {  # or gamma_inf, anti-shielding factors, for the ions in your material
    'Na': -4.0,
    'O':  -2.2
    # 'Mn': ...,  # add your own value here if you have one
}

# Formal ionic charges [units of e], only needed if compute_lattice_efg=True.
formal_charges = {
    'Na': +1, 'F': -1, 'Ca': +2, 'Li': +1, 'Cl': -1, 'Mg': +2, 'O': -2,
    'Al': +3,
    # add whatever your material contains
}

In [6]:
# Identify atom kinds

atm_kinds = get_atom_kinds(atoms)

print("\nAtom kinds:")
for kind, indices in atm_kinds.items():
    print(f"{kind}: {indices}")


print("\nO indices:")
print(atm_kinds["O"])


Atom kinds:
Al: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
O: [12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]

O indices:
[12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]


In [7]:
# O_idx   = [i for i, s in enumerate(atoms.get_chemical_symbols()) if s == 'O']
# Al_idx  = [i for i, s in enumerate(atoms.get_chemical_symbols()) if s == 'Al']

# test non zero spin O isotopes (e.g., 17^O, I=5/2, Q=-0.0265e-28 m^2, 0.038% abundance)
O_nuclear_spin   = 5/2
O_Quadrupole_moment = -0.0265e-28  # m^2 (0.038 abundance)


# test non zero spin Al isotopes (e.g., 27^Al, I=5/2, Q=+0.1466e-28 m^2, 100% abundance)
Al_I_spin   = 5/2
Al_Q_moment = +0.1466e-28  # m^2 (100 abundance)

Al2O3_charges = {'Al': +3, 'O': -2}

sphere_radius=50

Sternheimer antishielding factor = 0.0

In [8]:
probe_idx = atm_kinds["O"][4]
probe_pos = atoms.get_scaled_positions()[probe_idx]  

res = compute_efg(
    atoms, 
    probe_position=probe_pos, 
    atomic_charges=Al2O3_charges, 
    sphere_radius=sphere_radius,
    gamma_sternheimer=-0.0, 
    exclude_indices=(), 
    extra_charges=None,
    coords_are_cartesian=False, # True or False, depending on 'probe_position' type, frac. or cart.
    nuclear_spin=O_nuclear_spin,
    quadrupole_moment=O_Quadrupole_moment,
    verbose=True
)

Point-charge EFG: summing 61800 charges within radius 50.000 Å.

EFG analysis for atom 16 (O) at frac coord. (0.6936, 0.6936, 0.2500)
Vzz          = -1.17942258e+21 V/m^2
Vyy          =  9.52937763e+20 V/m^2
Vxx          =  2.26484815e+20 V/m^2
eta          =  0.61593950 (unitless)
chi_Q_MHz    =  0.75573524 MHz
nu_z_MHz     =  0.11336029 MHz
nu_Q_MHz     =  0.12031476 MHz

EFG tensor V_ab (V/m^2) =
----------------------------------------------------------------------
 [ -7.68149844e+18,  1.35195984e+20, -9.23031800e+20 ]
 [  1.35195984e+20,  1.48429377e+20,  5.32912658e+20 ]
 [ -9.23031800e+20,  5.32912658e+20, -1.40747879e+20 ]
----------------------------------------------------------------------
Trace(V_ab) =  4.09600e+05
Symmetric   = True

principal axes (unitless) = 
----------------------------------------------------------------------
 [ -5.00000000e-01,  6.20221174e-01, -6.04421786e-01 ]
 [ -8.66025404e-01, -3.58084862e-01,  3.48963081e-01 ]
 [ -7.52786722e-12, -6.97926162e-

Sternheimer antishielding factor = -2.2

In [9]:
probe_idx = atm_kinds["O"][4]
probe_pos = atoms.get_scaled_positions()[probe_idx]  
probe_pos = atoms.get_positions()[probe_idx] 

res = compute_efg(
    atoms, 
    probe_position=probe_pos, 
    atomic_charges=Al2O3_charges, 
    sphere_radius=sphere_radius,
    gamma_sternheimer=-2.2, 
    exclude_indices=(), 
    extra_charges=None,
    coords_are_cartesian=True, # True or False, depending on 'probe_position' type, frac. or cart.
    nuclear_spin=O_nuclear_spin,
    quadrupole_moment=O_Quadrupole_moment,
    verbose=True
)

Point-charge EFG: summing 61800 charges within radius 50.000 Å.



EFG analysis for atom 16 (O) at frac coord. (0.6936, 0.6936, 0.2500)
Vzz          = -3.77415225e+21 V/m^2
Vyy          =  3.04940084e+21 V/m^2
Vxx          =  7.24751407e+20 V/m^2
eta          =  0.61593950 (unitless)
chi_Q_MHz    =  2.41835278 MHz
nu_z_MHz     =  0.36275292 MHz
nu_Q_MHz     =  0.38500724 MHz

EFG tensor V_ab (V/m^2) =
----------------------------------------------------------------------
 [ -2.45807950e+19,  4.32627148e+20, -2.95370176e+21 ]
 [  4.32627148e+20,  4.74974006e+20,  1.70532051e+21 ]
 [ -2.95370176e+21,  1.70532051e+21, -4.50393211e+20 ]
----------------------------------------------------------------------
Trace(V_ab) =  1.24518e+06
Symmetric   = True

principal axes (unitless) = 
----------------------------------------------------------------------
 [ -5.00000000e-01,  6.20221174e-01, -6.04421786e-01 ]
 [ -8.66025404e-01, -3.58084862e-01,  3.48963081e-01 ]
 [ -7.52775620e-12, -6.97926162e-01, -7.16169723e-01 ]
------------------------------------------

lets build the same structure with `ase.crystal`

In [10]:
from ase.spacegroup import crystal

# Build Hexagonal Corundum (ie. alpha-Al2O3) using Space Group 167 (R-3c)
a, c = 4.754, 12.990
atoms = crystal(
    symbols=["Al", "O"],
    basis=[(0, 0, 0.35228), (0.3064, 0, 0.25)],
    spacegroup=167,
    cellpar=[a, a, c, 90, 90, 120],
)

# Atom kinds/symbols in atoms
atm_kinds = get_atom_kinds(atoms)

# Assign formal charges (Al: +3.0, O: -2.0)
charges = {"Al": 3.0, "O": -2.0}

# Define probe input properties
O_nuclear_spin   = 5/2
O_Quadrupole_moment = -0.0265e-28  # m^2 (0.038 abundance)
O_gamma_sternheimer = -2.2
O_atm_idx = atm_kinds["O"][2]
O_atm_pos = atoms.get_scaled_positions()[O_atm_idx]

# Compute bare EFG tensor
efg_tensor = point_charge_EFG(
            atoms=atoms,
            site_position=O_atm_pos,
            charges=charges,
            sphere_radius=50.0,
            coords_are_cartesian=False,  
            gamma_sternheimer=0,
            verbose=False,
        )

# or Compute EFG along with properties in dict format
results = compute_efg(
    atoms, 
    probe_position=O_atm_pos, 
    atomic_charges=charges, 
    sphere_radius=50.0,
    gamma_sternheimer=O_gamma_sternheimer, 
    coords_are_cartesian=False, 
    nuclear_spin=O_nuclear_spin,
    quadrupole_moment=O_Quadrupole_moment,
    verbose=False
)

results

{'Vxx': np.float64(7.247509699025467e+20),
 'Vyy': np.float64(3.0494015362241606e+21),
 'Vzz': np.float64(-3.7741525061267735e+21),
 'eta': 0.6159397540369369,
 'V_aa': array([ 7.24750970e+20,  3.04940154e+21, -3.77415251e+21]),
 'nu_z_MHz': np.float64(0.3627529412726441),
 'nu_Q_MHz': np.float64(0.38500728241417964),
 'chi_Q_MHz': 2.4183529418176284,
 'EFG_tensor': array([[-2.45809079e+19,  4.32626961e+20, -2.95370218e+21],
        [ 4.32626961e+20,  4.74973677e+20,  1.70532075e+21],
        [-2.95370218e+21,  1.70532075e+21, -4.50392769e+20]]),
 'principal_axes': array([[-5.00000000e-01,  6.20221153e-01, -6.04421807e-01],
        [-8.66025404e-01, -3.58084850e-01,  3.48963093e-01],
        [ 2.66453526e-15, -6.97926186e-01, -7.16169699e-01]]),
 'probe_index': 14,
 'probe_symbol': 'O',
 'probe_position': array([0.6936, 0.6936, 0.25  ])}

We can use `PointChargeEFG` calculator to do same as above:

In [11]:
# Initialize PointChargeEFG
calc = PointChargeEFG(
    atoms=atoms,
    charges=charges,
    sphere_radius=50.0,
)

In [12]:
# Calculate EFG at a position
efg_tensor = calc.get_raw_tensor(
    position=O_atm_pos,  # position (O_atm_pos) or index (O_atm_idx)
    coords_are_cartesian=False,
    gamma_sternheimer=O_gamma_sternheimer
    )

calc.print_summary()

  EFG CALCULATION SUMMARY: Site O (Index 14)

-- Site Metadata & Inputs --
  Fractional Pos : ( 0.69360,  0.69360,  0.25000)
  Cartesian Pos  : ( 1.64869,  2.85561,  3.24750)
  Sum Radius     : 50.0 Å
  Sternheimer G  : -2.2000

-- Raw EFG Tensor V_ij (V/m^2) --
  (  -2.458091e+19    4.326270e+20   -2.953702e+21 )
  (   4.326270e+20    4.749737e+20    1.705321e+21 )
  (  -2.953702e+21    1.705321e+21   -4.503928e+20 )


In [13]:
# EFG and other properties
results = calc.compute_at(
    position=O_atm_pos,  # position (O_atm_pos) or index (O_atm_idx)
    nuclear_spin=O_nuclear_spin,
    quadrupole_moment=O_Quadrupole_moment,
    coords_are_cartesian=False,
    gamma_sternheimer=O_gamma_sternheimer
    )

calc.print_summary()

  EFG CALCULATION SUMMARY: Site O (Index 14)

-- Site Metadata & Inputs --
  Fractional Pos : ( 0.69360,  0.69360,  0.25000)
  Cartesian Pos  : ( 1.64869,  2.85561,  3.24750)
  Sum Radius     : 50.0 Å
  Sternheimer G  : -2.2000
  Nuclear Spin I : 2.5
  Quadrupole Q   : -2.6500e-30 m^2

-- Raw EFG Tensor V_ij (V/m^2) --
  (  -2.458091e+19    4.326270e+20   -2.953702e+21 )
  (   4.326270e+20    4.749737e+20    1.705321e+21 )
  (  -2.953702e+21    1.705321e+21   -4.503928e+20 )

-- Principal Diagonal & Asymmetry --
  Vxx =  7.247510e+20 V/m^2 | Vyy =  3.049402e+21 V/m^2 | Vzz = -3.774153e+21 V/m^2
  Asymmetry Parameter (eta) :  0.61594

-- Quadrupolar Frequencies --
  Cq  (Quadrupolar Coupling)       :  2.418353 MHz
  nu_Q                             :  0.385007 MHz
  nu_z                             :  0.362753 MHz
